# GSE312129 donor one probe

Implements steps 1 and 2 of `RR_LABELING_PROTOCOL_2026-09-09.md`, and nothing else.

**This notebook does not score motifs.** It answers two questions cheaply, on a single
donor, before any bulk download happens:

1. Does the Cell Ranger ARC h5 actually contain a `Peaks` feature type? If not, the
   fragment files are needed after all and the whole plan changes.
2. Does the prespecified labeling rule recover a usable fibroblast population, against
   the failure thresholds fixed in section 4 of the protocol?

Every gate raises. None of them warn and continue.


## 1. Confirm the protocol was read

In [ ]:
# The protocol fixes the marker panels, the assignment rule, the failure thresholds
# and the paralog detectability floor. Nothing below is interpretable without it.
# Set this to the date line at the top of the protocol file.

PROTOCOL_READ = '<<SET ME>>'      # expected: '2026-09-09'

if PROTOCOL_READ != '2026-09-09':
    raise RuntimeError(
        'STOP: read RR_LABELING_PROTOCOL_2026-09-09.md first, then set PROTOCOL_READ '
        "to '2026-09-09'. The panels and thresholds below are fixed there and must not "
        'be edited here.')
print('protocol acknowledged')

## 2. Install and mount

In [ ]:
!pip -q install scanpy anndata h5py 2>&1 | tail -2
import scanpy as sc, anndata as ad, numpy as np, pandas as pd, os, urllib.request
from google.colab import drive
drive.mount('/content/drive')
WORK = '/content/drive/MyDrive/RR/runx_paralog'
os.makedirs(WORK, exist_ok=True)
sc.settings.verbosity = 1
print('scanpy', sc.__version__, '| WORK', WORK)

## 3. Download donor one only

Per GSM supplementary path, so the RAW tar and its spatial images are never touched.
SSC1 is GSM9338143. Section 7 of the protocol has the full map.

In [ ]:
GSM, SAMPLE = 'GSM9338143', 'SSC1'
fn  = f'{GSM}_{SAMPLE}_filtered_feature_bc_matrix.h5'
url = f'https://ftp.ncbi.nlm.nih.gov/geo/samples/GSM9338nnn/{GSM}/suppl/{fn}'
dst = f'{WORK}/{fn}'

if not os.path.exists(dst):
    print('downloading', url)
    urllib.request.urlretrieve(url, dst)
print(dst, f'{os.path.getsize(dst)/1e6:.1f} MB')
if os.path.getsize(dst) < 1_000_000:
    raise RuntimeError('STOP: file is implausibly small. Open it and look before going on.')

## 4. GATE: is `Peaks` in the h5?

Protocol section 6 assumes it is, and says so explicitly as an assumption. This is
where the assumption is tested. If it fails, the fragment files are required, the
per-donor peak set argument collapses, and the plan is rewritten before anything else.

In [ ]:
full = sc.read_10x_h5(dst, gex_only=False)
full.var_names_make_unique()

print('object:', full.shape)
if 'feature_types' not in full.var:
    raise RuntimeError('STOP: no feature_types in .var. This is not a multiome h5.')

counts = full.var['feature_types'].value_counts()
print(counts)

if 'Peaks' not in counts.index:
    raise RuntimeError(
        'GATE FAILED: no Peaks feature type. Protocol section 6 assumed Cell Ranger ARC '
        'output carrying both modalities. It does not.\n'
        'Consequence: the atac_fragments.tsv.gz files are required, a common peak set '
        'must be built, and the per-donor independence argument in section 6 no longer '
        'holds. Stop and rewrite the plan.')
if 'Gene Expression' not in counts.index:
    raise RuntimeError('GATE FAILED: no Gene Expression feature type. Cannot attribute a paralog.')

rna  = full[:, full.var.feature_types == 'Gene Expression'].copy()
atac = full[:, full.var.feature_types == 'Peaks'].copy()
assert (rna.obs_names == atac.obs_names).all(), 'barcodes not aligned'
print(f'\nPASSED. RNA {rna.shape}   ATAC {atac.shape}   same {rna.n_obs:,} nuclei')

## 5. QC and labeling, exactly as fixed in protocol sections 2 and 3

In [ ]:
PANELS = {
 'fibroblast'  : ['COL1A1','COL1A2','COL3A1','DCN','LUM','PDGFRA','FBLN1'],
 'keratinocyte': ['KRT14','KRT5','KRT1','KRT10','KRT6A'],
 'endothelial' : ['PECAM1','VWF','CDH5','EGFL7'],
 'pericyte_smc': ['RGS5','MYH11','NOTCH3','KCNJ8'],   # ACTA2/TAGLN deliberately absent
 'myeloid'     : ['LYZ','CD68','ITGAM','CSF1R','AIF1'],
 't_nk'        : ['CD3D','CD3E','IL7R','CD2','TRAC'],
 'b_plasma'    : ['MS4A1','CD79A','JCHAIN','MZB1'],
 'mast'        : ['TPSAB1','TPSB2','CPA3','MS4A2'],
 'melanocyte'  : ['PMEL','MLANA','TYRP1','DCT'],
 'adnexal'     : ['KRT7','AQP5','SCGB2A2','DCD'],
}
MIN_GENES, MAX_MT_PCT, MARGIN = 500, 5.0, 0.05

q = rna.copy()
q.var['mt'] = q.var_names.str.startswith('MT-')
sc.pp.calculate_qc_metrics(q, qc_vars=['mt'], inplace=True, percent_top=None, log1p=False)
before = q.n_obs
q = q[(q.obs.n_genes_by_counts >= MIN_GENES) & (q.obs.pct_counts_mt <= MAX_MT_PCT)].copy()
print(f'QC: {before:,} -> {q.n_obs:,} nuclei kept')

counts_layer = q.copy()                      # keep raw counts for detectability
sc.pp.normalize_total(q, target_sum=1e4); sc.pp.log1p(q)

for name, genes in PANELS.items():
    present = [g for g in genes if g in q.var_names]
    print(f'  {name:<14} {len(present)}/{len(genes)} present', '' if present else '  <-- EMPTY')
    if not present:
        raise RuntimeError(f'STOP: no genes of the {name} panel are in this object.')
    sc.tl.score_genes(q, present, score_name=f'score_{name}')

S = q.obs[[f'score_{n}' for n in PANELS]].to_numpy()
order = np.argsort(-S, axis=1)
top, second = S[np.arange(len(S)), order[:,0]], S[np.arange(len(S)), order[:,1]]
labels = np.array(list(PANELS))[order[:,0]]
labels[(top - second) < MARGIN] = 'ambiguous'
q.obs['label'] = pd.Categorical(labels)
print()
print(q.obs['label'].value_counts())

## 6. The failure thresholds from protocol section 4

In [ ]:
vc = q.obs['label'].value_counts()
n_fib  = int(vc.get('fibroblast', 0))
n_ker  = int(vc.get('keratinocyte', 0))
amb    = float(vc.get('ambiguous', 0)) / q.n_obs

print(f'fibroblast nuclei : {n_fib:,}')
print(f'keratinocyte      : {n_ker:,}')
print(f'ambiguous fraction: {amb:.1%}')

fib = counts_layer[q.obs['label'] == 'fibroblast'].copy()
mk  = [g for g in ['COL1A1','PDGFRA'] if g in fib.var_names]
X   = fib[:, mk].X
det = float(np.asarray((X > 0).sum(axis=1)).ravel().astype(bool).mean()) if fib.n_obs else 0.0
print(f'fibroblasts with COL1A1 or PDGFRA detected: {det:.1%}')

fails = []
if n_fib < 200:                       fails.append(f'fibroblast nuclei {n_fib} < 200 floor')
if amb > 0.40:                        fails.append(f'ambiguous {amb:.1%} > 40%')
if det < 0.30:                        fails.append(f'detection sanity {det:.1%} < 30%')
if n_ker and n_fib > 10 * n_ker:      fails.append(f'fibroblasts {n_fib} > 10x keratinocytes {n_ker}')

if fails:
    raise RuntimeError('LABELING FAILED on ' + SAMPLE + ':\n  ' + '\n  '.join(fails) +
                       '\nDo not proceed to the other donors. Report this.')
print('\nall section 4 thresholds passed for', SAMPLE)

## 7. The question most likely to end the study

Protocol section 5. A paralog is detectable in a donor if at least 5 percent of that
donor's fibroblast nuclei carry a nonzero count. RUNX2 is lowly expressed outside bone
and single nuclei data is sparse, so expect it to fail this.

In [ ]:
FLOOR = 0.05
print(f'{SAMPLE}: {fib.n_obs:,} fibroblast nuclei\n')
rows = []
for g in ['RUNX1','RUNX2','RUNX3','COL1A1','SMAD3','EGR1']:
    if g not in fib.var_names:
        rows.append((g, 'gene absent from object', ''))
        continue
    v = np.asarray(fib[:, g].X.todense()).ravel() if hasattr(fib[:, g].X,'todense') \
        else np.asarray(fib[:, g].X).ravel()
    frac = float((v > 0).mean())
    rows.append((g, f'{frac:.2%}', 'detectable' if frac >= FLOOR else 'BELOW FLOOR'))
print(pd.DataFrame(rows, columns=['gene','pct nuclei nonzero','verdict']).to_string(index=False))

print('\n--- what this means for donor one ---')
r2 = [r for r in rows if r[0] == 'RUNX2']
if r2 and r2[0][2] == 'BELOW FLOOR':
    print('RUNX2 is below the detectability floor in SSC1.')
    print('If that repeats in more than half the donors, the preregistered outcome is')
    print('"attribution impossible". That is a sparsity limit, not a biological null,')
    print('and it does not get written up as one.')
else:
    print('RUNX2 clears the floor in SSC1. Extend to the other thirteen donors.')

## 8. Stop here

This notebook ends without scoring a single motif, by design.

If sections 4, 6 and 7 all passed, the next step is the same pipeline across the
remaining thirteen donors, and only then motif deviations. If any of them raised, the
result is the raise, and it gets reported as a labeling or detectability outcome rather
than being worked around.